In [1]:
import pandas as pd

from database_utils import postgres_engine

In [2]:
query = "select * from raw.here__traffic_flows where Extract(month from data_queried_at) = 7"


con, engine = postgres_engine()


july=pd.read_sql_query(query, con=con)

In [3]:
import json
july["currentFlow.subSegments"] = july["currentFlow.subSegments"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
july_ex = july.explode("currentFlow.subSegments").reset_index(drop=True)
july_ex = pd.json_normalize(july_ex["currentFlow.subSegments"])

"""
data = pd.concat([july, july.explode('currentFlow.subSegments').reset_index(drop=True).drop(columns=['currentFlow.subSegments'])], 
                  axis=1)"""

"\ndata = pd.concat([july, july.explode('currentFlow.subSegments').reset_index(drop=True).drop(columns=['currentFlow.subSegments'])], \n                  axis=1)"

In [7]:
import numpy as np
from pyproj import Transformer
from shapely import wkt
from shapely.ops import substring, transform as shapely_transform

SEG = "currentFlow.subSegments"

# The sample coordinates are around Ann Arbor (UTM zone 17N).
to_meters = Transformer.from_crs(4326, 26917, always_xy=True).transform
to_lonlat = Transformer.from_crs(26917, 4326, always_xy=True).transform

parts = july.assign(_row_id=np.arange(len(july)))
parts = (
    parts.loc[parts[SEG].notna()]
         .explode(SEG)
         .copy()
)

parts["subsegment_length_m"] = parts[SEG].map(lambda segment: segment["length"])

# cumsum is grouped by the original row, so offsets restart for each route.
cum_length = parts.groupby("_row_id")["subsegment_length_m"].cumsum()
route_length = parts.groupby("_row_id")["subsegment_length_m"].transform("sum")

parts["start_fraction"] = (cum_length - parts["subsegment_length_m"]) / route_length
parts["end_fraction"] = cum_length / route_length

def slice_route(row):
    line = row["geometry"]
    if isinstance(line, str):
        line = wkt.loads(line)

    line_m = shapely_transform(to_meters, line)
    segment_m = substring(
        line_m,
        row["start_fraction"] * line_m.length,
        row["end_fraction"] * line_m.length,
    )
    return shapely_transform(to_lonlat, segment_m)

parts["subsegment_geometry"] = parts.apply(slice_route, axis=1)

In [8]:
parts

,location.description,location.length,location.shape.links,currentFlow.speed,currentFlow.speedUncapped,currentFlow.freeFlow,currentFlow.jamFactor,currentFlow.confidence,currentFlow.traversability,currentFlow.subSegments,...,confidence,traversability,junctionTraversability,jamTendency,data_queried_at,_row_id,subsegment_length_m,start_fraction,end_fraction,subsegment_geometry
6,S State Rd,3821.0,"[{""points"": [{""lat"": 42.22812, ""lng"": -83.7851...",16.666668,16.666668,18.055555,0.7,0.99,open,"{'length': 3777.0, 'speed': 16.666668, 'speedU...",...,None,None,None,None,2026-07-01 08:00:04.112,6,3777.0,0.000000,0.988485,"LINESTRING (-83.7851 42.22811999999999, -83.78..."
6,S State Rd,3821.0,"[{""points"": [{""lat"": 42.22812, ""lng"": -83.7851...",16.666668,16.666668,18.055555,0.7,0.99,open,"{'length': 44.0, 'speed': 8.611112, 'speedUnca...",...,None,None,None,None,2026-07-01 08:00:04.112,6,44.0,0.988485,1.000000,LINESTRING (-83.73924459819018 42.229311436938...
30,S State Rd,3821.0,"[{""points"": [{""lat"": 42.22812, ""lng"": -83.7851...",17.777779,17.777779,18.055555,0.2,0.99,open,"{'length': 3777.0, 'speed': 17.777779, 'speedU...",...,None,None,None,None,2026-07-01 07:30:04.266,30,3777.0,0.000000,0.988485,"LINESTRING (-83.7851 42.22811999999999, -83.78..."
30,S State Rd,3821.0,"[{""points"": [{""lat"": 42.22812, ""lng"": -83.7851...",17.777779,17.777779,18.055555,0.2,0.99,open,"{'length': 44.0, 'speed': 10.833334, 'speedUnc...",...,None,None,None,None,2026-07-01 07:30:04.266,30,44.0,0.988485,1.000000,LINESTRING (-83.73924459819018 42.229311436938...
35,W Stadium Blvd,2395.0,"[{""points"": [{""lat"": 42.27957, ""lng"": -83.7486...",NaN,NaN,10.833334,10.0,NaN,closed,"{'length': 10.0, 'speed': 8.055555, 'speedUnca...",...,None,None,None,None,2026-07-05 17:30:04.241,35,10.0,0.000000,0.004174,"LINESTRING (-83.7486 42.27957, -83.74872141947..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324231,US-23,2457.0,"[{""points"": [{""lat"": 42.3119, ""lng"": -83.66132...",14.444445,14.444445,18.888890,1.8,0.99,open,"{'length': 865.0, 'speed': 18.055555, 'speedUn...",...,None,None,None,None,2026-07-10 18:30:04.170,324231,865.0,0.000000,0.352199,"LINESTRING (-83.66132 42.31190000000001, -83.6..."
324231,US-23,2457.0,"[{""points"": [{""lat"": 42.3119, ""lng"": -83.66132...",14.444445,14.444445,18.888890,1.8,0.99,open,"{'length': 199.0, 'speed': 9.722222, 'speedUnc...",...,None,None,None,None,2026-07-10 18:30:04.170,324231,199.0,0.352199,0.433225,LINESTRING (-83.6716489017984 42.3111203460181...
324231,US-23,2457.0,"[{""points"": [{""lat"": 42.3119, ""lng"": -83.66132...",14.444445,14.444445,18.888890,1.8,0.99,open,"{'length': 196.0, 'speed': 18.055555, 'speedUn...",...,None,None,None,None,2026-07-10 18:30:04.170,324231,196.0,0.433225,0.513029,LINESTRING (-83.67386431964543 42.310391842302...
324231,US-23,2457.0,"[{""points"": [{""lat"": 42.3119, ""lng"": -83.66132...",14.444445,14.444445,18.888890,1.8,0.99,open,"{'length': 110.0, 'speed': 6.388889, 'speedUnc...",...,None,None,None,None,2026-07-10 18:30:04.170,324231,110.0,0.513029,0.557818,LINESTRING (-83.67606736100485 42.309710794723...


In [32]:
# Step 4: Concatenate the base fields with your new segment columns
df_final = pd.concat(
    [july_ex, july], axis=1
)

In [36]:
df_final[(df_final['location.description']=='Huron Pky')  & (df_final['data_queried_at'] == '2026-07-01 08:00:04.112000')]

,length,speed,speedUncapped,freeFlow,jamFactor,confidence,traversability,junctionTraversability,jamTendency,location.description,...,length,speed,speedUncapped,freeFlow,jamFactor,confidence,traversability,junctionTraversability,jamTendency,data_queried_at
38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
379,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
430,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
433,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
952,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huron Pky,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112


In [ ]:

july[(july['location.description']=='Huron Pky') & (july['data_queried_at'] == '2026-07-01 08:00:04.112000')]

,location.description,location.length,location.shape.links,currentFlow.speed,currentFlow.speedUncapped,currentFlow.freeFlow,currentFlow.jamFactor,currentFlow.confidence,currentFlow.traversability,currentFlow.subSegments,...,length,speed,speedUncapped,freeFlow,jamFactor,confidence,traversability,junctionTraversability,jamTendency,data_queried_at
38,Huron Pky,22.0,"[{""points"": [{""lat"": 42.27763, ""lng"": -83.6990...",7.777778,7.777778,10.833334,1.7,0.99,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
379,Huron Pky,1271.0,"[{""points"": [{""lat"": 42.30591, ""lng"": -83.6897...",8.888889,8.888889,16.388890,3.6,0.99,open,"[{""length"": 167.0, ""speed"": 9.722222, ""speedUn...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
416,Huron Pky,1830.0,"[{""points"": [{""lat"": 42.26359, ""lng"": -83.6782...",NaN,NaN,15.000000,10.0,NaN,closed,"[{""length"": 10.0, ""speed"": 12.777778, ""speedUn...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
430,Huron Pky,1280.0,"[{""points"": [{""lat"": 42.27288, ""lng"": -83.7107...",11.111112,16.388890,10.555556,0.0,0.95,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
433,Huron Pky,237.0,"[{""points"": [{""lat"": 42.27266, ""lng"": -83.6982...",10.555556,10.555556,10.555556,0.1,0.96,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
435,Huron Pky,1632.0,"[{""points"": [{""lat"": 42.28722, ""lng"": -83.7121...",13.611112,13.611112,15.555556,1.0,0.99,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
438,Huron Pky,19.0,"[{""points"": [{""lat"": 42.27763, ""lng"": -83.6990...",8.611112,8.611112,11.111112,1.1,0.98,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
439,Huron Pky,1716.0,"[{""points"": [{""lat"": 42.27458, ""lng"": -83.6789...",13.333334,13.333334,15.555556,1.1,0.99,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
440,Huron Pky,21.0,"[{""points"": [{""lat"": 42.27084, ""lng"": -83.6971...",10.555556,10.555556,11.944445,0.9,0.70,open,null,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
952,Huron Pky,2501.0,"[{""points"": [{""lat"": 42.29627, ""lng"": -83.7305...",11.666667,11.666667,14.722222,1.5,0.97,open,"[{""length"": 2202.0, ""speed"": 13.888889, ""speed...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112


In [28]:
july_ex[(july_ex['location.description']=='Huron Pky') & (july_ex['data_queried_at'] == '2026-07-01 08:00:04.112000')]

,location.description,location.length,location.shape.links,currentFlow.speed,currentFlow.speedUncapped,currentFlow.freeFlow,currentFlow.jamFactor,currentFlow.confidence,currentFlow.traversability,currentFlow.subSegments,...,length,speed,speedUncapped,freeFlow,jamFactor,confidence,traversability,junctionTraversability,jamTendency,data_queried_at
42,Huron Pky,22.0,"[{""points"": [{""lat"": 42.27763, ""lng"": -83.6990...",7.777778,7.777778,10.833334,1.7,0.99,open,None,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
500,Huron Pky,1271.0,"[{""points"": [{""lat"": 42.30591, ""lng"": -83.6897...",8.888889,8.888889,16.388890,3.6,0.99,open,"{'length': 167.0, 'speed': 9.722222, 'speedUnc...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
501,Huron Pky,1271.0,"[{""points"": [{""lat"": 42.30591, ""lng"": -83.6897...",8.888889,8.888889,16.388890,3.6,0.99,open,"{'length': 150.0, 'speed': 3.888889, 'speedUnc...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
502,Huron Pky,1271.0,"[{""points"": [{""lat"": 42.30591, ""lng"": -83.6897...",8.888889,8.888889,16.388890,3.6,0.99,open,"{'length': 775.0, 'speed': 16.38889, 'speedUnc...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
503,Huron Pky,1271.0,"[{""points"": [{""lat"": 42.30591, ""lng"": -83.6897...",8.888889,8.888889,16.388890,3.6,0.99,open,"{'length': 180.0, 'speed': 4.7222223, 'speedUn...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
551,Huron Pky,1830.0,"[{""points"": [{""lat"": 42.26359, ""lng"": -83.6782...",NaN,NaN,15.000000,10.0,NaN,closed,"{'length': 10.0, 'speed': 12.777778, 'speedUnc...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
552,Huron Pky,1830.0,"[{""points"": [{""lat"": 42.26359, ""lng"": -83.6782...",NaN,NaN,15.000000,10.0,NaN,closed,"{'length': 1612.0, 'freeFlow': 15.0, 'jamFacto...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
553,Huron Pky,1830.0,"[{""points"": [{""lat"": 42.26359, ""lng"": -83.6782...",NaN,NaN,15.000000,10.0,NaN,closed,"{'length': 208.0, 'speed': 12.777778, 'speedUn...",...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
576,Huron Pky,1280.0,"[{""points"": [{""lat"": 42.27288, ""lng"": -83.7107...",11.111112,16.388890,10.555556,0.0,0.95,open,None,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
581,Huron Pky,237.0,"[{""points"": [{""lat"": 42.27266, ""lng"": -83.6982...",10.555556,10.555556,10.555556,0.1,0.96,open,None,...,None,None,None,None,None,None,None,None,None,2026-07-01 08:00:04.112
